# 🚀 ZAIC 2026 DATAMATE — PIPELINE HUẤN LUYỆN LORA & TRIỂN KHAI SLM TRÊN GPU A100

> **Dự án:** Zalo AI Challenge 2026 — Track 2: DataMate  
> **Mô hình nền:** `Qwen/Qwen2.5-Coder-7B-Instruct`  
> **Mục tiêu:** Huấn luyện LoRA Adapter trên 761 mẫu SQL đã được kiểm chứng 100% ground-truth, sau đó gộp mô hình (Model Merge) và mở API Server phục vụ suy luận Hybrid Fast-Slow Path.

---

### 1. Kiểm tra Cấu hình Phần cứng GPU A100

In [ ]:
!nvidia-smi

### 2. Cài đặt Môi trường & Gỡ bỏ Xung đột Thư viện

In [ ]:
# 1. Dọn dẹp tiến trình cũ
!pkill -9 -f uvicorn || true
!pkill -9 -f cloudflared || true

# 2. Gỡ bỏ torchao (xung đột phiên bản cũ trên Colab) và cài đặt thư viện chuẩn
!pip uninstall -y torchao
!pip install -q trl peft transformers accelerate datasets fastapi uvicorn pydantic

# 3. Tải công cụ cloudflared mở đường hầm
!curl -fsSL https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -o /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared

import torch, gc
gc.collect()
torch.cuda.empty_cache()
print(f"✅ Môi trường đã sẵn sàng! PyTorch CUDA: {torch.cuda.is_available()}")

### 3. Nạp Tập Dữ Liệu Huấn luyện SFT (`train_sft.jsonl`)
> **Lưu ý:** Kéo thả file `train_sft.jsonl` từ máy tính vào khung Files (📁) bên trái trước khi chạy ô này.

In [ ]:
from datasets import load_dataset

DATASET_PATH = "/content/train_sft.jsonl"
dataset = load_dataset("json", data_files=DATASET_PATH, split="train")
print(f"📦 Đã nạp thành công {len(dataset)} mẫu huấn luyện SFT đã kiểm chứng 100%!")

### 4. Huấn luyện LoRA Adapter trên GPU A100 (~25 phút)
* 3 Epochs, effective batch size 16 (4 batch x 4 grad accum)
* Learning rate 2e-4 cosine scheduler
* LoRA Rank 16, Alpha 32

In [ ]:
import torch, gc
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, TaskType
from trl import SFTTrainer, SFTConfig

MODEL_ID = "Qwen/Qwen2.5-Coder-7B-Instruct"
print(f"⏳ Đang nạp mô hình nền {MODEL_ID}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True
)

# Cấu hình LoRA Adapter (Rank 16, Alpha 32)
peft_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    bias="none"
)

OUTPUT_DIR = "/content/qwen2.5-coder-7b-lora-datamate"

training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    logging_steps=10,
    save_strategy="epoch",
    bf16=True,
    max_length=2048,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    peft_config=peft_config,
    processing_class=tokenizer,
    args=training_args,
)

print("🚀 Bắt đầu quá trình huấn luyện LoRA SFT trên GPU A100...")
trainer.train()

# Lưu Adapter
trainer.model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"\n🎉 HUẤN LUYỆN HOÀN TẤT! Adapter đã được lưu tại: {OUTPUT_DIR}")

# Giải phóng VRAM của trainer
del trainer, model
gc.collect()
torch.cuda.empty_cache()

### 5. Gộp LoRA vào Mô hình Nền (Model Merge & Unload)
> Gộp vĩnh viễn LoRA weights vào base model trên CPU RAM để tạo thành mô hình độc lập 15GB, không phụ thuộc vào thư viện LoRA khi suy luận.

In [ ]:
import gc, torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

BASE_MODEL = "Qwen/Qwen2.5-Coder-7B-Instruct"
LORA_PATH = "/content/qwen2.5-coder-7b-lora-datamate"
MERGED_OUTPUT = "/content/qwen2.5-coder-7b-datamate"

print("⏳ Đang gộp LoRA Adapter vào mô hình nền trên CPU RAM (khoảng 45s)...\n")
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.bfloat16,
    device_map="cpu"
)

lora_model = PeftModel.from_pretrained(base_model, LORA_PATH)
merged_model = lora_model.merge_and_unload()

print("💾 Đang lưu mô hình hoàn chỉnh 15GB xuống đĩa...")
merged_model.save_pretrained(MERGED_OUTPUT)
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
tokenizer.save_pretrained(MERGED_OUTPUT)

del merged_model, lora_model, base_model
gc.collect()
print(f"🎉 GỘP THÀNH CÔNG 100%! Mô hình hoàn chỉnh tại: {MERGED_OUTPUT}")

### 6. Khởi động API Server Chuẩn OpenAI (FastAPI + PyTorch) & Mở Cloudflare Tunnel
> Server chạy đơn tiến trình siêu ổn định trên GPU A100, phục vụ chuẩn `/v1/chat/completions` cho Antigravity Agent.

In [ ]:
import subprocess, time, re, os, threading, urllib.request, torch
from fastapi import FastAPI
from pydantic import BaseModel
from transformers import AutoModelForCausalLM, AutoTokenizer
import uvicorn

!pkill -9 -f uvicorn || true
!pkill -9 -f cloudflared || true

MODEL_DIR = "/content/qwen2.5-coder-7b-datamate"
print(f"⏳ Đang nạp mô hình 15GB vào GPU A100 bằng PyTorch gốc...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_DIR,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)
print("✅ Mô hình đã nạp 100% vào GPU A100 sẵn sàng!\n")

# Tạo API Server chuẩn OpenAI
app = FastAPI()

class ChatRequest(BaseModel):
    messages: list
    max_tokens: int = 1024
    temperature: float = 0.0
    model: str = "datamate"

@app.get("/v1/models")
def get_models():
    return {"data": [{"id": "datamate"}, {"id": MODEL_DIR}]}

@app.post("/v1/chat/completions")
def chat_completions(req: ChatRequest):
    text = tokenizer.apply_chat_template(req.messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to("cuda")
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=req.max_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )
    output_text = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    return {"choices": [{"message": {"content": output_text}}]}

def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="warning")

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()
time.sleep(3)

# Mở Cloudflare Tunnel
print("🌐 Đang kết nối đường hầm Cloudflare Tunnel...")
cf_proc = subprocess.Popen(["cloudflared", "tunnel", "--url", "http://localhost:8000"], stderr=subprocess.PIPE, text=True)

tunnel_url = None
start_time = time.time()
while time.time() - start_time < 30:
    line = cf_proc.stderr.readline()
    m = re.search(r"https://([a-zA-Z0-9-]+\.trycloudflare\.com)", line)
    if m:
        tunnel_url = m.group(1)
        break
    time.sleep(0.1)

if tunnel_url:
    print("\n" + "="*70)
    print("🎉 API ENDPOINT PYTORCH SẴN SÀNG HOÀN TOÀN!")
    print(f"👉 Base URL: https://{tunnel_url}/v1")
    print("👉 Model: datamate")
    print("="*70 + "\n")
else:
    print("❌ Chưa bắt được link Cloudflare, hãy kiểm tra lại kết nối mạng.")